# Deep Research

### By: Bruno Conterato

## 0. Imports and Config


In [1]:
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    AsyncOpenAI,
    ModelSettings,
)
from pydantic import BaseModel
from pydantic.dataclasses import dataclass
import asyncio

In [2]:
@dataclass
class CONFIG:
    model: str = "qwen2.5"
    num_search_terms: int = 3
    debug_mode: bool = False

    query_example: str = """Sabemos que recrutadores de RH hoje utilizam bots ou agentes de inteligência artificial para otimizar a busca e seleção de perfis de candidatos a vagas no LinkedIn e currículos. Quero que faça uma busca e descubra os principais critérios de otimização de currículos que os tornem mais visíveis ou "encontráveis" por IAs de recrutadores de RH, para o meu cargo do LinkedIn: "DEEP LEARNING ENGINEER | APPLIED AI, INTELLIGENT SYSTEMS, AI AGENTS"."""

## 1. Tools and Definitions


### 1.1. Model to run Ollama

In [3]:
ollama_client = AsyncOpenAI(base_url="http://localhost:11434/v1")

model = OpenAIChatCompletionsModel(model=CONFIG.model, openai_client=ollama_client)

### 1.2. The tool for web search

In [4]:
import httpx


@function_tool
async def search_web(query: str) -> str:
    """Search the web for information about a given query."""

    async with httpx.AsyncClient() as client:
        try:
            response = await client.get(
                "http://localhost:4479/search/text", params={"query": query}
            )
            response.raise_for_status()
            results_arr = [
                f"Title: {r.title}\nBody: {r.body}" for r in response.json()["results"]
            ]
            return "\n\n".join(results_arr)
        except Exception as e:
            return f"Search failed: {str(e)}"

## 2. The Agents Definitions

### 2.1. The search planner Agent

In [5]:
from pydantic import Field


class WebSearchItem(BaseModel):
    reason: str = Field(
        description="Your reasoning for why this search is important to the query."
    )

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description="A list of web searches to perform to best answer the query."
    )


planner_agent = Agent(
    name="Planner",
    instructions=f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {CONFIG.num_search_terms} terms to query for.",
    model=model,
    output_type=WebSearchPlan,
)

In [6]:
# results = await Runner.run(planner_agent, CONFIG.query_example)

In [7]:
# results.final_output

### 2.2. The search Agent

In [8]:
search_agent = Agent(
    name="Search agent",
    instructions="You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself.",
    tools=[search_web],
    model=model,
    model_settings=ModelSettings(tool_choice="required"),
)

In [9]:
# results = await Runner.run(
#     search_agent, CONFIG.query_example
# )

In [10]:
# results.final_output

### 2.3. Writter Agent

In [11]:
class ReportData(BaseModel):
    short_summary: str = Field(
        description="A short 2-3 sentence summary of the findings."
    )

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(
        description="Suggested topics to research further"
    )


writer_agent = Agent(
    name="WriterAgent",
    instructions="You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words.",
    model=model,
    output_type=ReportData,
)

## 3. Agent Runner Functions


In [12]:
async def plan_searches(query: str) -> WebSearchPlan:
    if CONFIG.debug_mode:
        print("Planning searches...")
        print(f"User query: {query}")
    result = await Runner.run(planner_agent, f"Query: {query}")
    if CONFIG.debug_mode:
        print(f"Will perform searches: {result.final_output}")
    return result.final_output


async def process_search_query(item: WebSearchItem) -> str:
    search_result = await Runner.run(search_agent, item.query)
    return search_result.final_output


async def perform_searches(searchPlan: WebSearchPlan) -> str:
    if CONFIG.debug_mode:
        print("\nPerforming Searches...")
        print(f"Search plan: {searchPlan}")
    tasks = [asyncio.create_task(process_search_query(s)) for s in searchPlan.searches]
    results = await asyncio.gather(*tasks)
    response = ""
    for i, result in enumerate(results):
        response += f"Resultado da busca {i+1}:\n{result}"
    return response


async def write_report(info: str) -> ReportData:
    result = await Runner.run(writer_agent, info)
    return result.final_output

In [13]:
async def deep_research(query: str) -> ReportData:
    searchPlan: WebSearchPlan = await plan_searches(query)
    searchResults = await perform_searches(searchPlan)
    report = await write_report(searchResults)
    return report

In [14]:
result = await deep_research(CONFIG.query_example)

In [15]:
from IPython.display import display, Markdown

display(Markdown(result.markdown_report))

# Optimizing Resumes Using AI: A Comprehensive Guide

## Table of Contents
1. Introduction
2. Resume Optimization Strategies Utilizing AI
3. Deep Learning Engineer Job Requirements on LinkedIn
4. How AI Recruitment Tools Analyze and Prioritize Resumes
5. Conclusion
6. Recommendations
7. Future Directions in AI and Recruitment

## 1. Introduction

The integration of artificial intelligence (AI) in human resource management has significantly transformed the hiring process. One of the most impactful applications is the optimization of resumes to enhance their visibility, relevance, and suitability for various job roles. This report aims to explore effective resume optimization strategies using AI, with a particular focus on the requirements for deep learning engineer positions as observed on LinkedIn, and how AI tools analyze and prioritize resumes.

## 2. Resume Optimization Strategies Utilizing AI

### Highlighting Skills and Experiences

AI tools can help candidates tailor their resumes by identifying and emphasizing key skills and experiences relevant to the job description. Candidates should ensure that their resumes not only mention these critical points but also highlight how they have applied them in previous roles or projects. For instance, if a candidate has experience with Python programming, they could outline specific projects where this skill was utilized effectively.

### Ensuring Relevance to Job Descriptions

To improve the visibility of candidates' resumes during AI scans, it is essential to match the language and terminology used in the resume closely with that found in job listings. Utilizing keywords relevant to the profession or company, such as specific frameworks (TensorFlow, PyTorch), programming languages (Python, R), or technical terms from mathematical fields like 'linear algebra' or 'calculus,' can significantly enhance relevance.

### Consistent Formatting and Regular Updates

A well-formatted resume that adheres to a consistent structure is more likely to be recognized by AI systems. This consistency includes proper formatting of sections (Education, Professional Experience, Skills) and ensuring uniformity in font style, size, and alignment. Additionally, updating the resume regularly with new skills, projects, or jobs can keep it current and aligned with evolving industry trends.

### Customized Sections for Specific Roles/Industries

For specialized positions like deep learning engineers, customized sections that align with industry standards are crucial. This might include highlighting specific contributions to open-source projects, mentioning any involvement in academic research, or detailing significant achievements related to artificial intelligence (AI) and machine learning.

### Avoiding Overly Technical Language for Non-Technical Positions

While technical proficiency is critical for roles like deep learning engineer, applying overly complex language may alienate non-technical positions on resumes. Balancing the use of professional jargon with a plain-language summary of core competencies can make it more accessible and appealing to a broader audience.

### Including Measurable Achievements

AI tools often prioritize candidates based on measurable achievements. Candidates should incorporate quantifiable results from their past projects, such as improvements in accuracy or efficiency metrics, to demonstrate the tangible impact they have had. These achievements provide context for skills listed and help AI systems rank resumes more accurately.

## 3. Deep Learning Engineer Job Requirements on LinkedIn

### Programming Languages and Frameworks

Deep learning engineers typically require proficiency in programming languages such as Python. Knowledge of additional languages like R, along with a deep understanding of popular machine learning frameworks such as TensorFlow or PyTorch, is highly valued by employers.

### Strong Mathematical Background

A robust mathematical background forms the foundation for successful deep learning engineering roles. Skills in linear algebra, probability theory, statistics, and calculus enable engineers to develop and optimize models effectively. Employers look for candidates who can perform not just coding but also rigorous mathematical analysis required by complex machine learning algorithms.

### Relevant Academic Degrees

Although many roles do not mandatorily require advanced degrees, having a master's or doctoral degree in computer science, data science, mathematics, or related fields is beneficial. Such qualifications validate the candidate’s theoretical knowledge and practical skills pertinent to deep learning applications.

### Practical Experience Through Projects/Role

Experience with hands-on projects and real-world applications is crucial. Portfolios showcasing previous work on machine learning models can significantly bolster a deep learning engineer's resume. Highlighting participation in hackathons, contributing to open-source repositories, or leading projects that yielded meaningful outcomes are all valuable additions.

### Soft Skills

Soft skills play an essential role in collaborative environments common in technology sectors. Problem-solving abilities, adaptability, and the capacity to communicate complex ideas effectively are highly sought after by deep learning engineers' recruiters. Demonstrating leadership qualities, teamwork experience, and industry certifications can further fortify a candidacy.

### Specialized Qualifications and Certification

Specific certifications from professional bodies such as Google AI or Microsoft Azure validate both technical prowess and commitment to staying current with technological advancements. Earning certificates in specialized areas like reinforcement learning (RL) or deep neural networks (DNNs) can differentiate candidates and make them more attractive prospects for companies requiring cutting-edge expertise.

## 4. How AI Recruitment Tools Analyze and Prioritize Resumes

### Key Factors Evaluated by AI

AI recruitment tools evaluate resumes based on various factors including years of experience, education level, skills, work experience, and the relevance of qualifications to specific job requirements. For instance, a doctorate in computer science may be more valuable for certain roles than a bachelor’s degree.

### Sentiment Analysis and Cultural Fit

Some AI tools incorporate sentiment analysis to understand not just technical proficiency but also cultural fit within teams. This approach helps gauge whether candidates align with the organization's culture, values, and communication style. Positive attitudes, good interpersonal skills, and cultural competence are important factors in selection processes beyond hard data.

### Integration of Candidate History Data

Modern AI systems integrate historical application metrics from previous hiring rounds to predict future success rates accurately. By analyzing patterns in data across multiple candidates, these algorithms can identify common traits among successful employees, helping refine the selection process and enhance overall organizational performance.

### Machine Learning Models

Machine learning models train on large datasets containing examples of successful hires. These models learn to recognize key attributes associated with job success and use this information to prioritize potential candidates effectively. Regular training updates ensure that these algorithms keep up with industry trends and evolving role requirements, ensuring more accurate shortlisting of suitable applicants.

## 5. Conclusion

Optimizing resumes for roles like deep learning engineers in the age of AI-driven recruitment requires careful attention to both technical competencies and user-friendly presentation styles. By understanding what AI tools prioritize—such as relevant skills, measurable achievements, and cultural fit—it is possible to craft a highly effective resume that stands out.

## 6. Recommendations

### Tailored Approach

Crafting a tailored resume for each application remains crucial even in the sophisticated world of AI recruitment tools. Each job description has unique nuances; hence personalization ensures your resume aligns better with the specific criteria laid out by potential employers.

### Frequent Updates

Regularly updating your resume to reflect current skills and recent experiences will keep it relevant and competitive over time. This habit not only enhances your visibility but also demonstrates ongoing engagement and interest in professional growth.

### Online Presence

Maintaining an active online presence through LinkedIn profiles, personal blogs, or tech-driven forums can complement traditional resumes. Sharing insights on technical challenges and solutions can help establish credibility among potential employers while showcasing industry knowledge.

## 7. Future Directions in AI and Recruitment

As technology continues to evolve, so too will the methods employed for resume optimization. Emerging trends suggest a shift towards more personalized candidate interactions made possible through machine learning advancements. Additionally, greater emphasis on diversity and inclusion may lead to enhanced tools designed specifically to mitigate bias during hiring processes.

In summary, embracing AI strategies in optimizing resumes while maintaining human intuition provides an optimal blend of precision and personalization required for navigating today's competitive job market.

---
Note: This report is based on the provided data points and aims to offer a comprehensive guide for individuals seeking to enhance their visibility during resume screening processes facilitated by AI systems.